In [1]:
"""
Degradation Asymmetry Analysis
================================
Quantify the directionality of mRNA degradation within operon containment
clusters from PacBio FLNC isoform data (Syn1 minimal cell).

Hypothesis: 5'→3' exoribonuclease (RNase J1/J2) degradation is faster than
3'→5' exoribonuclease (RNase R / YhaM) degradation. If true, downstream
cleavage products are cleared quickly, and the surviving truncated isoforms
should predominantly share the original TSS (same 5' end) with shorter 3' ends.

Method:
  For each operon containment cluster, identify the longest isoform as the
  "full-length reference". Every shorter member is classified by comparing
  its 5' and 3' endpoints to the reference (within BOUNDARY_TOL):

    ┌─────────────────────────────────────────────────────────────────────┐
    │  Category             │ 5' end       │ 3' end       │ Implication  │
    │───────────────────────┼──────────────┼──────────────┼──────────────│
    │  same_5p_shorter_3p   │ ≈ reference  │ < reference  │ 3' erosion   │
    │  diff_5p_same_3p      │ > reference  │ ≈ reference  │ 5' erosion   │
    │  diff_5p_shorter_3p   │ > reference  │ < reference  │ both trimmed │
    │  other                │ edge cases   │              │              │
    └─────────────────────────────────────────────────────────────────────┘

  (Coordinates are described for + strand; − strand is mirrored.)

  If 5'→3' exo outpaces 3'→5' exo:
    - Downstream endo-cleavage products (which carry a 5'-monophosphate,
      the ideal RNase J substrate) are rapidly destroyed
    - Upstream products (sharing the original TSS) persist → "same_5p_shorter_3p"
      dominates

Output:
  - Per-category isoform counts and read-weighted fractions
  - Per-operon degradation ratio (3' erosion reads / 5' erosion reads)
  - Summary statistics and publication-ready figures
"""

'\nDegradation Asymmetry Analysis\n================================\nQuantify the directionality of mRNA degradation within operon containment\nclusters from PacBio FLNC isoform data (Syn1 minimal cell).\n\nHypothesis: 5\'→3\' exoribonuclease (RNase J1/J2) degradation is faster than\n3\'→5\' exoribonuclease (RNase R / YhaM) degradation. If true, downstream\ncleavage products are cleared quickly, and the surviving truncated isoforms\nshould predominantly share the original TSS (same 5\' end) with shorter 3\' ends.\n\nMethod:\n  For each operon containment cluster, identify the longest isoform as the\n  "full-length reference". Every shorter member is classified by comparing\n  its 5\' and 3\' endpoints to the reference (within BOUNDARY_TOL):\n\n    ┌─────────────────────────────────────────────────────────────────────┐\n    │  Category             │ 5\' end       │ 3\' end       │ Implication  │\n    │───────────────────────┼──────────────┼──────────────┼──────────────│\n    │  same_5p_

## Incompleteness of Isoforms

In [ ]:

import sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration — must match the operon segmentation notebook
# ═══════════════════════════════════════════════════════════════════════════════
MOTHER_FOLDER = ".."
ISOFORMS_TSV  = MOTHER_FOLDER + "/isoform_annotation/isoform_clusters_annotated.tsv"
OUT_FOLDER    = MOTHER_FOLDER + "/Operon_Annotation_Visualization"

MIN_READS     = 50    # same threshold as the segmentation notebook
BOUNDARY_TOL  = 10    # bp tolerance for "same endpoint" classification

# ═══════════════════════════════════════════════════════════════════════════════
# Step 1 — Load isoforms and re-run containment clustering
# ═══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv(ISOFORMS_TSV, sep="\t")
df_iso = df[df["n_reads"] >= MIN_READS].copy()
print(f"Isoforms loaded (n_reads >= {MIN_READS}): {len(df_iso)}")


def cluster_isoforms_with_members(isoforms: pd.DataFrame,
                                   tol: int = BOUNDARY_TOL) -> list[dict]:
    """
    Containment clustering (identical to the segmentation notebook).
    Returns cluster dicts that include full member-level detail.
    """
    if isoforms.empty:
        return []

    iso    = isoforms.reset_index(drop=True)
    starts = iso["start0"].astype(int).tolist()
    ends   = iso["end0"].astype(int).tolist()
    n      = len(iso)

    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        parent[find(x)] = find(y)

    order = sorted(range(n), key=lambda i: starts[i])
    for idx, i in enumerate(order):
        si, ei = starts[i], ends[i]
        for j in order[idx + 1:]:
            sj = starts[j]
            if sj >= ei + tol:
                break
            ej = ends[j]
            i_in_j = (si >= sj - tol) and (ei <= ej + tol)
            j_in_i = (sj >= si - tol) and (ej <= ei + tol)
            if i_in_j or j_in_i:
                union(i, j)

    components = defaultdict(list)
    for i in range(n):
        components[find(i)].append(i)

    strand = iso.loc[0, "strand"]
    blocks = []
    for indices in components.values():
        members = []
        for k in indices:
            members.append({
                "isoform_id": iso.loc[k, "isoform_id"],
                "start0":     int(iso.loc[k, "start0"]),
                "end0":       int(iso.loc[k, "end0"]),
                "n_reads":    int(iso.loc[k, "n_reads"]),
                "length":     int(iso.loc[k, "end0"] - iso.loc[k, "start0"]),
            })
        # Identify the longest member as the full-length reference
        members.sort(key=lambda m: m["length"], reverse=True)
        blocks.append({
            "strand":  strand,
            "start0":  min(m["start0"] for m in members),
            "end0":    max(m["end0"]   for m in members),
            "members": members,
        })
    return sorted(blocks, key=lambda b: b["start0"])


clusters_plus  = cluster_isoforms_with_members(df_iso[df_iso["strand"] == "+"])
clusters_minus = cluster_isoforms_with_members(df_iso[df_iso["strand"] == "-"])
all_clusters = clusters_plus + clusters_minus
print(f"Containment clusters: + strand {len(clusters_plus)}, "
      f"- strand {len(clusters_minus)}, total {len(all_clusters)}")

# ═══════════════════════════════════════════════════════════════════════════════
# Step 2 — Classify truncated isoforms within each cluster
# ═══════════════════════════════════════════════════════════════════════════════
#
# For + strand:  TSS = start0 (5' end, low coord),  TTS = end0  (3' end, high coord)
# For − strand:  TSS = end0   (5' end, high coord), TTS = start0 (3' end, low coord)
#
# "Same 5'" means the isoform's 5' end matches the reference within tol.
# "Shorter 3'" means the isoform's 3' end is receded inward from the reference.

records = []

for cluster in all_clusters:
    strand  = cluster["strand"]
    members = cluster["members"]
    if len(members) < 2:
        continue  # singleton clusters have no truncated isoforms

    ref = members[0]  # longest isoform = full-length reference

    if strand == "+":
        ref_5p = ref["start0"]  # TSS
        ref_3p = ref["end0"]    # TTS
    else:
        ref_5p = ref["end0"]    # TSS (high coord for minus)
        ref_3p = ref["start0"]  # TTS (low coord for minus)

    for mem in members[1:]:  # all non-reference (shorter) members
        if strand == "+":
            mem_5p = mem["start0"]
            mem_3p = mem["end0"]
            same_5p = abs(mem_5p - ref_5p) <= BOUNDARY_TOL
            same_3p = abs(mem_3p - ref_3p) <= BOUNDARY_TOL
            shorter_3p = mem_3p < ref_3p - BOUNDARY_TOL
            shorter_5p = mem_5p > ref_5p + BOUNDARY_TOL  # 5' receded inward
        else:
            mem_5p = mem["end0"]
            mem_3p = mem["start0"]
            same_5p = abs(mem_5p - ref_5p) <= BOUNDARY_TOL
            same_3p = abs(mem_3p - ref_3p) <= BOUNDARY_TOL
            shorter_3p = mem_3p > ref_3p + BOUNDARY_TOL   # for minus, 3' is low coord
            shorter_5p = mem_5p < ref_5p - BOUNDARY_TOL   # for minus, 5' is high coord

        # Classify
        if same_5p and shorter_3p:
            category = "same_5p_shorter_3p"
        elif shorter_5p and same_3p:
            category = "diff_5p_same_3p"
        elif shorter_5p and shorter_3p:
            category = "diff_5p_shorter_3p"
        elif same_5p and same_3p:
            category = "same_both"       # near-identical to reference
        else:
            category = "other"

        # Compute how many bp are trimmed from each end
        if strand == "+":
            trim_3p = max(0, ref_3p - mem_3p)
            trim_5p = max(0, mem_5p - ref_5p)
        else:
            trim_3p = max(0, mem_3p - ref_3p)
            trim_5p = max(0, ref_5p - mem_5p)

        records.append({
            "strand":       strand,
            "ref_isoform":  ref["isoform_id"],
            "ref_length":   ref["length"],
            "ref_reads":    ref["n_reads"],
            "mem_isoform":  mem["isoform_id"],
            "mem_length":   mem["length"],
            "mem_reads":    mem["n_reads"],
            "category":     category,
            "trim_5p_bp":   trim_5p,
            "trim_3p_bp":   trim_3p,
            "frac_retained": round(mem["length"] / ref["length"], 3),
        })

trunc_df = pd.DataFrame(records)
print(f"\nTruncated isoforms classified: {len(trunc_df)}")

# ═══════════════════════════════════════════════════════════════════════════════
# Step 3 — Summary statistics
# ═══════════════════════════════════════════════════════════════════════════════

# --- 3a: Isoform counts by category ---
cat_counts = trunc_df["category"].value_counts()
cat_reads  = trunc_df.groupby("category")["mem_reads"].sum()
total_iso  = len(trunc_df)
total_reads = trunc_df["mem_reads"].sum()

print("\n" + "="*70)
print("TRUNCATION CATEGORY SUMMARY")
print("="*70)
print(f"{'Category':<25} {'Isoforms':>10} {'%':>8} {'Reads':>12} {'%':>8}")
print("-"*70)
for cat in ["same_5p_shorter_3p", "diff_5p_same_3p", "diff_5p_shorter_3p",
            "same_both", "other"]:
    n_iso  = cat_counts.get(cat, 0)
    n_read = cat_reads.get(cat, 0)
    print(f"{cat:<25} {n_iso:>10,} {n_iso/total_iso*100:>7.1f}% "
          f"{n_read:>12,} {n_read/total_reads*100:>7.1f}%")
print("-"*70)
print(f"{'TOTAL':<25} {total_iso:>10,} {'100.0%':>8} "
      f"{total_reads:>12,} {'100.0%':>8}")


# --- 3b: Degradation asymmetry ratio ---
reads_3p_erosion = cat_reads.get("same_5p_shorter_3p", 0)
reads_5p_erosion = cat_reads.get("diff_5p_same_3p", 0)
if reads_5p_erosion > 0:
    asymmetry_ratio = reads_3p_erosion / reads_5p_erosion
    print(f"\n3' erosion / 5' erosion read ratio: {asymmetry_ratio:.2f}")
    print(f"  → {asymmetry_ratio:.1f}× more reads in 3'-eroded isoforms")
    print(f"  → Consistent with faster 5'→3' exo clearing downstream fragments")
else:
    asymmetry_ratio = float("inf")
    print(f"\n3' erosion / 5' erosion read ratio: inf (no 5' erosion isoforms)")

n_3p_erosion = cat_counts.get("same_5p_shorter_3p", 0)
n_5p_erosion = cat_counts.get("diff_5p_same_3p", 0)
if n_5p_erosion > 0:
    iso_ratio = n_3p_erosion / n_5p_erosion
    print(f"\n3' erosion / 5' erosion isoform count ratio: {iso_ratio:.2f}")
else:
    iso_ratio = float("inf")
    print(f"\n3' erosion / 5' erosion isoform count ratio: inf")


# --- 3c: Per-strand breakdown ---
print("\n" + "-"*70)
print("Per-strand breakdown:")
for strand in ["+", "-"]:
    sub = trunc_df[trunc_df["strand"] == strand]
    n3 = sub[sub["category"] == "same_5p_shorter_3p"].shape[0]
    n5 = sub[sub["category"] == "diff_5p_same_3p"].shape[0]
    nb = sub[sub["category"] == "diff_5p_shorter_3p"].shape[0]
    r3 = sub[sub["category"] == "same_5p_shorter_3p"]["mem_reads"].sum()
    r5 = sub[sub["category"] == "diff_5p_same_3p"]["mem_reads"].sum()
    ratio_str = f"{r3/r5:.2f}" if r5 > 0 else "inf"
    print(f"  {strand} strand: 3'erosion={n3} ({r3:,} reads), "
          f"5'erosion={n5} ({r5:,} reads), both={nb}, "
          f"ratio={ratio_str}")


# ═══════════════════════════════════════════════════════════════════════════════
# Step 4 — Trim-length distributions (how far is each end eroded?)
# ═══════════════════════════════════════════════════════════════════════════════

# For 3'-eroded isoforms: distribution of 3' trim length
erosion_3p = trunc_df[trunc_df["category"] == "same_5p_shorter_3p"]["trim_3p_bp"]
erosion_5p = trunc_df[trunc_df["category"] == "diff_5p_same_3p"]["trim_5p_bp"]

print("\n" + "-"*70)
print("Trim-length distributions (bp eroded from reference):")
if len(erosion_3p) > 0:
    print(f"\n  3' erosion (same_5p_shorter_3p), n={len(erosion_3p)}:")
    print(f"    median = {erosion_3p.median():.0f} bp")
    print(f"    mean   = {erosion_3p.mean():.0f} bp")
    print(f"    Q25    = {erosion_3p.quantile(0.25):.0f} bp")
    print(f"    Q75    = {erosion_3p.quantile(0.75):.0f} bp")
if len(erosion_5p) > 0:
    print(f"\n  5' erosion (diff_5p_same_3p), n={len(erosion_5p)}:")
    print(f"    median = {erosion_5p.median():.0f} bp")
    print(f"    mean   = {erosion_5p.mean():.0f} bp")
    print(f"    Q25    = {erosion_5p.quantile(0.25):.0f} bp")
    print(f"    Q75    = {erosion_5p.quantile(0.75):.0f} bp")


# ═══════════════════════════════════════════════════════════════════════════════
# Step 5 — Per-operon degradation ratio
# ═══════════════════════════════════════════════════════════════════════════════
# For each cluster with ≥2 truncated isoforms, compute:
#   degradation_ratio = reads(same_5p_shorter_3p) / reads(diff_5p_same_3p)
# A ratio > 1 means 3' erosion dominates (consistent with faster 5'→3' exo).

operon_ratios = []
for cluster in all_clusters:
    ref_id = cluster["members"][0]["isoform_id"] if cluster["members"] else None
    sub = trunc_df[trunc_df["ref_isoform"] == ref_id]
    if len(sub) < 2:
        continue
    r3 = sub[sub["category"] == "same_5p_shorter_3p"]["mem_reads"].sum()
    r5 = sub[sub["category"] == "diff_5p_same_3p"]["mem_reads"].sum()
    n3 = (sub["category"] == "same_5p_shorter_3p").sum()
    n5 = (sub["category"] == "diff_5p_same_3p").sum()
    if r3 + r5 > 0:
        operon_ratios.append({
            "ref_isoform":  ref_id,
            "strand":       cluster["strand"],
            "n_members":    len(cluster["members"]),
            "reads_3p_erosion": r3,
            "reads_5p_erosion": r5,
            "n_3p_erosion": n3,
            "n_5p_erosion": n5,
            "ratio":        r3 / r5 if r5 > 0 else r3/1, 
            "dominant":     "3p_erosion" if r3 > r5 else ("5p_erosion" if r5 > r3 else "balanced"),
        })

ratio_df = pd.DataFrame(operon_ratios)
finite_ratios = ratio_df[ratio_df["ratio"] != -float("inf")]
positive_ratios = finite_ratios[finite_ratios["ratio"] > 0]

print("\n" + "="*70)
print("PER-OPERON DEGRADATION ASYMMETRY")
print("="*70)
if len(ratio_df) > 0:
    n_3p_dom  = (ratio_df["dominant"] == "3p_erosion").sum()
    n_5p_dom  = (ratio_df["dominant"] == "5p_erosion").sum()
    n_balanced = (ratio_df["dominant"] == "balanced").sum()
    n_inf     = (ratio_df["ratio"] == float("inf")).sum()
    print(f"Operons analysed: {len(ratio_df)}")
    print(f"  3' erosion dominant: {n_3p_dom}  ({n_3p_dom/len(ratio_df)*100:.1f}%)")
    print(f"  5' erosion dominant: {n_5p_dom}  ({n_5p_dom/len(ratio_df)*100:.1f}%)")
    print(f"  Balanced:            {n_balanced}")
    print(f"  Ratio = inf (no 5' erosion at all): {n_inf}")
    if len(finite_ratios) > 0:
        print(f"\n  Finite ratio distribution (n={len(finite_ratios)}):")
        print(f"    median = {finite_ratios['ratio'].median():.2f}")
        print(f"    mean   = {finite_ratios['ratio'].mean():.2f}")
        print(f"    Q25    = {finite_ratios['ratio'].quantile(0.25):.2f}")
        print(f"    Q75    = {finite_ratios['ratio'].quantile(0.75):.2f}")


# ═══════════════════════════════════════════════════════════════════════════════
# Step 6 — Figures
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Degradation Asymmetry Analysis — Syn1 Operon Isoforms",
             fontsize=13, fontweight="bold", y=0.98)

# --- Panel A: Category bar chart (isoform counts + read-weighted) ---
ax = axes[0, 0]
cats = ["same_5p_shorter_3p", "diff_5p_same_3p", "diff_5p_shorter_3p", "same_both", "other"]
labels = ["Same 5', shorter 3'\n(3' erosion)",
          "Diff 5', same 3'\n(5' erosion)",
          "Diff 5', shorter 3'\n(both trimmed)",
          "Same both\n(near-identical)",
          "Other"]
colors = ["#2166AC", "#B2182B", "#762A83", "#969696", "#CCCCCC"]

iso_vals  = [cat_counts.get(c, 0) for c in cats]
read_vals = [cat_reads.get(c, 0) for c in cats]
iso_pcts  = [v / total_iso * 100 for v in iso_vals]
read_pcts = [v / total_reads * 100 for v in read_vals]

x = np.arange(len(cats))
w = 0.35
bars1 = ax.bar(x - w/2, iso_pcts,  w, color=colors, alpha=0.85, edgecolor="white", label="By isoform count")
bars2 = ax.bar(x + w/2, read_pcts, w, color=colors, alpha=0.50, edgecolor="white", hatch="//", label="By read count")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7.5, ha="center")
ax.set_ylabel("Percentage (%)", fontsize=9)
ax.set_title("A. Truncation categories", fontsize=10, fontweight="bold")
ax.legend(fontsize=7.5, loc="upper right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))

# Add count annotations on bars
for bar, val in zip(bars1, iso_vals):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"n={val:,}", ha="center", va="bottom", fontsize=6.5)

# --- Panel B: Trim-length distributions ---
ax = axes[0, 1]
bins = np.arange(0, min(5000, max(
    erosion_3p.max() if len(erosion_3p) > 0 else 0,
    erosion_5p.max() if len(erosion_5p) > 0 else 0
) + 200), 50)

if len(erosion_3p) > 0:
    ax.hist(erosion_3p, bins=bins, alpha=0.7, color="#2166AC",
            label=f"3' erosion (n={len(erosion_3p):,})", density=True)
if len(erosion_5p) > 0:
    ax.hist(erosion_5p, bins=bins, alpha=0.7, color="#B2182B",
            label=f"5' erosion (n={len(erosion_5p):,})", density=True)
ax.set_xlabel("Bases trimmed (bp)", fontsize=9)
ax.set_ylabel("Density", fontsize=9)
ax.set_title("B. Trim-length distributions", fontsize=10, fontweight="bold")
ax.legend(fontsize=8)
# --- Panel C: Per-operon ratio distribution ---
ax = axes[1, 0]
if len(positive_ratios) > 0:
    log_ratios = np.log2(positive_ratios["ratio"].values)
    ax.hist(log_ratios, bins=40, color="#4393C3", alpha=0.8, edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", linewidth=1.2,
               label="Balanced (ratio=1)")
    med = np.median(log_ratios)
    ax.axvline(med, color="black", linestyle="-", linewidth=1.5,
               label=f"Median = {2**med:.2f}× (log₂ = {med:.2f})")
    ax.set_xlabel("log₂(3' erosion reads / 5' erosion reads)", fontsize=9)
    ax.set_ylabel("Number of operons", fontsize=9)
    ax.legend(fontsize=7.5)
ax.set_title("C. Per-operon degradation asymmetry", fontsize=10, fontweight="bold")
ax.set_title("C. Per-operon degradation asymmetry", fontsize=10, fontweight="bold")
# Add annotation for operons with only 3' erosion (ratio = inf)
n_inf = (ratio_df["ratio"] == float("inf")).sum()
if n_inf > 0:
    ax.text(0.98, 0.95, f"+{n_inf} operons with\nonly 3' erosion (ratio=∞)",
            transform=ax.transAxes, fontsize=7.5, ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#E0E0E0", alpha=0.8))

# --- Panel D: Read fraction retained vs category ---
ax = axes[1, 1]
for cat, color, label in [
    ("same_5p_shorter_3p", "#2166AC", "3' erosion"),
    ("diff_5p_same_3p",    "#B2182B", "5' erosion"),
    ("diff_5p_shorter_3p", "#762A83", "Both trimmed"),
]:
    sub = trunc_df[trunc_df["category"] == cat]
    if len(sub) > 0:
        ax.scatter(sub["frac_retained"], sub["mem_reads"],
                   alpha=0.3, s=8, color=color, label=f"{label} (n={len(sub):,})")
ax.set_xlabel("Fraction of full-length retained", fontsize=9)
ax.set_ylabel("Read count (truncated isoform)", fontsize=9)
ax.set_yscale("log")
ax.set_title("D. Read support vs. length retention", fontsize=10, fontweight="bold")
ax.legend(fontsize=7.5, markerscale=2)

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("degradation_asymmetry.pdf", dpi=300, bbox_inches="tight")
fig.savefig("degradation_asymmetry.png", dpi=200, bbox_inches="tight")
plt.close()
print("\nFigures saved: degradation_asymmetry.pdf / .png")


# ═══════════════════════════════════════════════════════════════════════════════
# Step 7 — Save detailed tables
# ═══════════════════════════════════════════════════════════════════════════════
trunc_df.to_csv("truncation_classification.tsv", sep="\t", index=False)
print(f"Saved: truncation_classification.tsv ({len(trunc_df)} rows)")

if len(ratio_df) > 0:
    ratio_df.to_csv("per_operon_degradation_ratio.tsv", sep="\t", index=False)
    print(f"Saved: per_operon_degradation_ratio.tsv ({len(ratio_df)} rows)")

print("\nDone.")

Isoforms loaded (n_reads >= 50): 4064
Containment clusters: + strand 148, - strand 168, total 316

Truncated isoforms classified: 3748

TRUNCATION CATEGORY SUMMARY
Category                    Isoforms        %        Reads        %
----------------------------------------------------------------------
same_5p_shorter_3p             1,306    34.8%    1,088,003    59.0%
diff_5p_same_3p                  974    26.0%      342,523    18.6%
diff_5p_shorter_3p             1,169    31.2%      330,073    17.9%
same_both                          7     0.2%        4,201     0.2%
other                            292     7.8%       80,112     4.3%
----------------------------------------------------------------------
TOTAL                          3,748   100.0%    1,844,912   100.0%

3' erosion / 5' erosion read ratio: 3.18
  → 3.2× more reads in 3'-eroded isoforms
  → Consistent with faster 5'→3' exo clearing downstream fragments

3' erosion / 5' erosion isoform count ratio: 1.34

---------------

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# tmRNA (ssrA) as a fraction of non-rRNA expression
# ═══════════════════════════════════════════════════════════════════════════════
# Motivation: In E. coli, tmRNA accounts for ~25% of non-rRNA reads, reflecting
# heavy ribosome-rescue demand. If Syn1 shows a similar or higher fraction it
# would be consistent with RNase Y endo-cleavages constantly generating truncated
# mRNAs that stall ribosomes.
#
# We use the Illumina avg_sense_TPM as the primary abundance metric (averaged
# over the three biological replicates already stored in the CSV), with
# PacBio_sense_TPM shown alongside for comparison.
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd

TPM_CSV = "../Transcriptomics_Quantification/syn1_Illumina_PacBio_TPM_profiles.csv"

tpm = pd.read_csv(TPM_CSV)

# ── classify genes ──────────────────────────────────────────────────────────
is_rRNA   = tpm["rna_type"] == "rRNA"
is_tmRNA  = tpm["rna_type"] == "tmRNA"          # ssrA / MMSYN1_0158
is_non_rRNA = ~is_rRNA                           # everything that is not rRNA

# ── Illumina avg_sense_TPM ──────────────────────────────────────────────────
illumina_col   = "avg_sense_TPM"
pacbio_col     = "PacBio_sense_TPM"

total_illumina       = tpm[illumina_col].sum()
non_rRNA_illumina    = tpm.loc[is_non_rRNA, illumina_col].sum()
tmRNA_illumina       = tpm.loc[is_tmRNA,    illumina_col].sum()

rRNA_illumina        = tpm.loc[is_rRNA,     illumina_col].sum()

total_pacbio         = tpm[pacbio_col].sum()
non_rRNA_pacbio      = tpm.loc[is_non_rRNA, pacbio_col].sum()
tmRNA_pacbio         = tpm.loc[is_tmRNA,    pacbio_col].sum()
rRNA_pacbio          = tpm.loc[is_rRNA,     pacbio_col].sum()

tmRNA_frac_illumina  = tmRNA_illumina  / non_rRNA_illumina  * 100
tmRNA_frac_pacbio    = tmRNA_pacbio    / non_rRNA_pacbio    * 100
rRNA_frac_illumina   = rRNA_illumina   / total_illumina     * 100
rRNA_frac_pacbio     = rRNA_pacbio     / total_pacbio       * 100

print("=" * 62)
print("tmRNA (ssrA) ABUNDANCE — Syn1 vs E. coli reference")
print("=" * 62)

# Show the tmRNA row for transparency
tmrna_row = tpm[is_tmRNA][["locus_tag", "gene_name", "rna_type",
                             illumina_col, pacbio_col]]
print("\ntmRNA gene entry:")
print(tmrna_row.to_string(index=False))

print(f"\n{'Metric':<45} {'Illumina':>10} {'PacBio':>10}")
print("-" * 65)
print(f"{'rRNA fraction of total TPM':<45} {rRNA_frac_illumina:>9.1f}% {rRNA_frac_pacbio:>9.1f}%")
print(f"{'non-rRNA TPM (sum)':<45} {non_rRNA_illumina:>10.1f} {non_rRNA_pacbio:>10.1f}")
print(f"{'tmRNA TPM':<45} {tmRNA_illumina:>10.1f} {tmRNA_pacbio:>10.1f}")
print(f"{'tmRNA / non-rRNA TPM  (Syn1)':<45} {tmRNA_frac_illumina:>9.1f}% {tmRNA_frac_pacbio:>9.1f}%")
print(f"{'tmRNA / non-rRNA reads (E. coli reference)':<45} {'~25 %':>10}")

print("\nInterpretation:")
ecoli_ref = 25.0
for label, frac in [("Illumina", tmRNA_frac_illumina), ("PacBio", tmRNA_frac_pacbio)]:
    fold = frac / ecoli_ref
    direction = "HIGHER" if frac > ecoli_ref else "lower"
    print(f"  {label}: tmRNA = {frac:.1f}% of non-rRNA TPM  "
          f"({fold:.2f}× vs E. coli ~25%) — {direction} than E. coli")

print("\n  A fraction ≥ 25% is consistent with elevated ribosome-rescue")
print("  demand, as expected if RNase Y endo-cleavages frequently generate")
print("  truncated mRNAs that stall ribosomes and require tmRNA tagging.")

# ═══════════════════════════════════════════════════════════════════════════════
# Ribosome-trapping potential: 3'-eroded mRNAs as a fraction of all mRNA reads
# ═══════════════════════════════════════════════════════════════════════════════
# A 3'-eroded isoform (same 5' end, truncated 3' end) lacks the stop codon.
# Ribosomes translating such transcripts reach the 3' end without terminating
# and become stalled — exactly the substrates that tmRNA rescues.
#
# Estimate: (reads from same_5p_shorter_3p isoforms) / (all classified reads)
# Both numerator and denominator come from the PacBio containment-clustering
# analysis performed above (trunc_df / cat_reads / total_reads).
# ═══════════════════════════════════════════════════════════════════════════════

# reads_3p_erosion and total_reads are already defined from Step 3 above.
# For a more conservative denominator we also include full-length reference reads.
total_reads_incl_ref = trunc_df["mem_reads"].sum() + \
    sum(c["members"][0]["n_reads"] for c in all_clusters if c["members"])

frac_no_stop_truncated = reads_3p_erosion / total_reads * 100
frac_no_stop_all       = reads_3p_erosion / total_reads_incl_ref * 100

print("\n" + "=" * 62)
print("RIBOSOME-TRAPPING POTENTIAL — 3'-eroded (stop-codon-less) mRNAs")
print("=" * 62)
print(f"\n  3'-eroded reads (same_5p_shorter_3p) : {reads_3p_erosion:>12,}")
print(f"  Total truncated isoform reads         : {total_reads:>12,}")
print(f"  Total reads incl. full-length refs    : {total_reads_incl_ref:>12,}")
print(f"\n  % of truncated reads lacking stop codon : {frac_no_stop_truncated:.1f}%")
print(f"  % of all isoform reads lacking stop codon: {frac_no_stop_all:.1f}%")
print(f"\n  Interpretation:")
print(f"  At least {frac_no_stop_all:.1f}% of PacBio isoform reads represent transcripts")
print(f"  without a stop codon. These are the direct substrates for ribosome")
print(f"  stalling and tmRNA-mediated rescue, corroborating the high tmRNA")
print(f"  abundance observed above.")


tmRNA (ssrA) ABUNDANCE — Syn1 vs E. coli reference

tmRNA gene entry:
  locus_tag gene_name rna_type  avg_sense_TPM  PacBio_sense_TPM
MMSYN1_0158      ssrA    tmRNA     24234.1491          249.6743

Metric                                          Illumina     PacBio
-----------------------------------------------------------------
rRNA fraction of total TPM                          6.6%       0.0%
non-rRNA TPM (sum)                              926065.1   979490.9
tmRNA TPM                                        24234.1      249.7
tmRNA / non-rRNA TPM  (Syn1)                        2.6%       0.0%
tmRNA / non-rRNA reads (E. coli reference)         ~25 %

Interpretation:
  Illumina: tmRNA = 2.6% of non-rRNA TPM  (0.10× vs E. coli ~25%) — lower than E. coli
  PacBio: tmRNA = 0.0% of non-rRNA TPM  (0.00× vs E. coli ~25%) — lower than E. coli

  A fraction ≥ 25% is consistent with elevated ribosome-rescue
  demand, as expected if RNase Y endo-cleavages frequently generate
  truncated mRNAs